# 选修E1 · Day 2：Agent框架对比 · 上机练习（v5.0）

> **真实库**：LangGraph（真实运行）+ CrewAI/AutoGen（静态API对比）
> **核心对比**：ReAct vs Plan-Execute | LangGraph vs CrewAI vs AutoGen
> **营销映射**：同一个营销任务（透肌精华竞品分析+策略），不同框架实现对比

本笔记本包含 **6个TODO填空**，完成后你将：
1. 定义营销工具和离线StubLLM
2. 用`create_react_agent`构建LangGraph ReAct Agent
3. 用`StateGraph`构建LangGraph Plan-Execute Agent
4. 运行ReAct和Plan-Execute，对比步数/调用/输出
5. 用CrewAI API结构编写等价实现（静态对比设计哲学）
6. 用AutoGen API结构编写等价实现 + 生成四框架对比表

> 📦 真实库说明见 `data/README.md`
> 📖 理论讲义见 `notes.md`


In [ ]:
# === 导入真实库 ===
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# === 真实营销数据（基于护肤品电商场景，复用Day 1）===
PRODUCT_DB = {
    "透肌精华": "透肌焕亮精华液，299元，主打美白焕亮，含烟酰胺3%+维C衍生物，目标用户25-35岁都市白领。",
    "玻尿酸面霜": "玻尿酸保湿面霜，159元，主打深层补水，含双重玻尿酸，目标用户18-30岁女性。",
}
COMPETITOR_DB = {
    "雅诗兰黛": "雅诗兰黛小棕瓶精华，760元/30ml，市场占有率18%，优势：品牌力强、渠道完善；劣势：价格高、年轻化不足。",
    "兰蔻": "兰蔻小黑瓶精华，780元/30ml，市场占有率15%，优势：科技感强、专柜体验；劣势：下沉市场覆盖弱。",
}

# === 离线模拟LLM（无需API Key，预编排工具调用序列）===
class StubChatModel(BaseChatModel):
    """离线模拟LLM，预编排工具调用序列，保证无API Key可运行。
    替换为ChatOpenAI/ChatAnthropic即可使用真实LLM。"""
    responses: list = []
    call_index: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        idx = self.call_index
        self.call_index += 1
        if idx < len(self.responses):
            resp = self.responses[idx]
        else:
            resp = AIMessage(content="任务完成。")
        return ChatResult(generations=[ChatGeneration(message=resp)])

    @property
    def _llm_type(self):
        return "stub"

    def bind_tools(self, tools, **kwargs):
        return self

# 统一的营销任务（所有框架用同一个任务对比）
MARKETING_TASK = "为透肌精华制定营销策略，竞品分析雅诗兰黛，并写入策略文件"

print("真实库导入成功")
print(f"  产品库: {list(PRODUCT_DB.keys())}")
print(f"  竞品库: {list(COMPETITOR_DB.keys())}")
print(f"  统一营销任务: {MARKETING_TASK}")
print(f"  StubChatModel: 离线模式（无API Key可运行）")
print(f"  crewai/autogen: 未安装 -> 采用静态API结构对比（不阻塞）")


---
## TODO1：用@tool装饰器定义营销工具

工具是Agent的"手"。在LangChain中，用`@tool`装饰器定义工具。
工具的**名称、docstring、参数类型**就是LLM看到的"接口契约"。

这些工具将被LangGraph的ReAct Agent和Plan-Execute Agent共用，
保证同一个营销任务在不同框架下的工具调用一致。

需要定义三个营销工具：
1. `search_product_info(product_name)` - 搜索产品信息（使用PRODUCT_DB）
2. `analyze_competitor(competitor_name)` - 分析竞品策略（使用COMPETITOR_DB）
3. `write_strategy(filename, content)` - 将策略写入文件

> 参考教材 § Day 1 四、工具使用（工具定义复用Day 1，保证任务一致性）


In [ ]:
# TODO1: 用@tool装饰器定义三个营销工具

@tool
def search_product_info(product_name: str) -> str:
    """搜索产品信息。参数: product_name - 产品名称。返回产品详情（价格、功效、目标用户）。"""
    return PRODUCT_DB.get(product_name, f"未找到'{product_name}'的产品信息。可用产品: {list(PRODUCT_DB.keys())}")

@tool
def analyze_competitor(competitor_name: str) -> str:
    """分析竞争对手。参数: competitor_name - 竞品名称。返回竞品分析（价格、市占率、优劣势）。"""
    return COMPETITOR_DB.get(competitor_name, f"未找到'{competitor_name}'的竞品信息。可分析竞品: {list(COMPETITOR_DB.keys())}")

@tool
def write_strategy(filename: str, content: str) -> str:
    """将营销策略写入文件。参数: filename - 文件名; content - 策略内容。返回写入确认。"""
    return f"策略已写入 {filename}，共 {len(content)} 字。"

# 验证工具
tools = [search_product_info, analyze_competitor, write_strategy]
print(f"定义了 {len(tools)} 个营销工具:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

# 直接测试工具（不经过Agent）
result = search_product_info.invoke({"product_name": "透肌精华"})
print(f"\n工具测试: search_product_info('透肌精华') -> {result}")


---
## TODO2：用create_react_agent构建LangGraph ReAct Agent

ReAct（Reasoning + Acting）的核心循环：
```
Thought -> Action -> Observation -> Thought -> ... -> FINISH
```

用LangGraph的`create_react_agent`构建ReAct Agent：
- model: 使用StubChatModel（预编排工具调用序列）
- tools: 使用TODO1定义的三个工具
- prompt: 系统提示，定义Agent角色

预编排轨迹模拟LLM的Thought-Action决策：搜索产品 -> 分析竞品 -> 写策略 -> 完成

> 参考教材 § Day 1 三、ReAct范式（本Day用同一范式做框架对比基准）


In [ ]:
# TODO2: 用create_react_agent构建LangGraph ReAct Agent

# 预编排ReAct轨迹：模拟LLM的Thought-Action决策序列
react_trajectory = [
    AIMessage(content="", tool_calls=[{"name": "search_product_info", "args": {"product_name": "透肌精华"}, "id": "c1"}]),
    AIMessage(content="", tool_calls=[{"name": "analyze_competitor", "args": {"competitor_name": "雅诗兰黛"}, "id": "c2"}]),
    AIMessage(content="", tool_calls=[{"name": "write_strategy", "args": {"filename": "strategy_react.txt", "content": "差异化策略：主打性价比+年轻化定位。透肌精华299元vs雅诗兰黛760元，价格优势明显。目标用户25-35岁都市白领，注重性价比和成分透明。"}, "id": "c3"}]),
    AIMessage(content="营销策略已制定完成。基于产品分析（299元，美白焕亮，含烟酰胺3%）和竞品分析（760元，品牌力强但价格高），制定差异化策略：主打性价比+年轻化定位，已写入strategy_react.txt。"),
]

react_model = StubChatModel(responses=react_trajectory)

# 构建ReAct Agent（LangGraph预构建）
react_agent = create_react_agent(
    react_model,
    tools,
    prompt="你是一个营销策略Agent。根据用户需求，依次调用工具：搜索产品信息、分析竞品、撰写并写入策略。"
)

print("LangGraph ReAct Agent构建成功")
print(f"  框架: LangGraph (create_react_agent)")
print(f"  模型: {react_model._llm_type} (离线StubLLM)")
print(f"  工具: {[t.name for t in tools]}")
print(f"  预编排轨迹: {len(react_trajectory)} 步")
print(f"  设计哲学: Agent即图（ReAct循环隐式编排）")


---
## TODO3：用StateGraph构建LangGraph Plan-Execute Agent

Plan-Execute与ReAct的核心区别：
- **ReAct**：边推理边执行，每步可根据观测调整下一步
- **Plan-Execute**：先一次性规划所有步骤，再顺序执行

用LangGraph的`StateGraph`实现：
1. 定义`PlanExecuteState`（TypedDict）：task, plan, current_step, results, final_answer
2. `plan_node`：一次性生成完整计划（4步）
3. `execute_node`：根据当前步骤描述执行
4. `should_continue`：条件函数，判断是否还有未执行步骤
5. 构建图：plan -> execute -> (continue: execute | end: END)

> 参考教材 § Day 2 一、LangGraph设计哲学（Agent即图，显式控制流）


In [ ]:
# TODO3: 用StateGraph构建LangGraph Plan-Execute Agent

class PlanExecuteState(TypedDict):
    task: str
    plan: list
    current_step: int
    results: list
    final_answer: str

def plan_node(state):
    """规划节点：一次性生成完整计划（不依赖LLM，模拟Plan阶段）"""
    task = state["task"]
    plan = [
        f"搜索产品信息: {task}",
        f"分析竞品策略: {task}",
        "撰写差异化策略",
        "写入策略文件",
    ]
    return {"plan": plan, "current_step": 0, "results": []}

def execute_node(state):
    """执行节点：执行当前步骤（顺序执行，不可动态调整）"""
    step_idx = state["current_step"]
    plan = state["plan"]
    if step_idx >= len(plan):
        return {"final_answer": "所有步骤已完成"}
    step_desc = plan[step_idx]
    if "搜索产品" in step_desc:
        result = PRODUCT_DB.get("透肌精华", "未找到")
    elif "分析竞品" in step_desc:
        result = COMPETITOR_DB.get("雅诗兰黛", "未找到")
    elif "撰写" in step_desc:
        result = "差异化策略：主打性价比+年轻化定位"
    elif "写入" in step_desc:
        result = "策略已写入 strategy_plan_execute.txt"
    else:
        result = f"执行: {step_desc}"
    return {
        "results": state["results"] + [result],
        "current_step": step_idx + 1,
    }

def should_continue(state):
    if state["current_step"] < len(state["plan"]):
        return "continue"
    return "end"

# 构建Plan-Execute图（显式定义节点和边）
pe_graph = StateGraph(PlanExecuteState)
pe_graph.add_node("plan", plan_node)
pe_graph.add_node("execute", execute_node)
pe_graph.set_entry_point("plan")
pe_graph.add_edge("plan", "execute")
pe_graph.add_conditional_edges("execute", should_continue, {
    "continue": "execute",
    "end": END,
})
plan_execute_agent = pe_graph.compile()

print("LangGraph Plan-Execute Agent构建成功")
print(f"  框架: LangGraph (StateGraph)")
print(f"  节点: plan, execute")
print(f"  边: plan -> execute -> (continue: execute | end: END)")
print(f"  设计哲学: Agent即图（显式控制流，条件分支）")
print(f"  与ReAct对比: 先规划{4}步，再顺序执行（不可动态调整）")


---
## TODO4：运行ReAct和Plan-Execute，对比执行轨迹

同一个营销任务，两种模式实跑，对比：
- 工具调用次数
- 模型调用次数（ReAct）/ 步骤数（Plan-Execute）
- 输出质量
- 执行模式差异

> 天道推演视角：ReAct的因果链在每步涌现，Plan-Execute的因果链在Plan阶段确定


In [ ]:
# TODO4: 运行ReAct和Plan-Execute，对比执行轨迹

# === 运行ReAct Agent ===
print("=" * 70)
print("【实跑】LangGraph ReAct Agent")
print("=" * 70)
react_result = react_agent.invoke({"messages": [("user", MARKETING_TASK)]})

react_tool_calls = 0
react_thoughts = 0
print(f"\n任务: {MARKETING_TASK}")
for i, msg in enumerate(react_result["messages"]):
    msg_type = type(msg).__name__
    if msg_type == "HumanMessage":
        print(f"  [Step {i}] Human: {msg.content}")
    elif msg_type == "AIMessage":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                react_tool_calls += 1
                print(f"  [Step {i}] Thought -> Action: {tc['name']}({tc['args']})")
        if msg.content:
            react_thoughts += 1
            print(f"  [Step {i}] AI: {msg.content}")
    elif msg_type == "ToolMessage":
        print(f"  [Step {i}] Observation: {msg.content}")

# === 运行Plan-Execute Agent ===
print("\n" + "=" * 70)
print("【实跑】LangGraph Plan-Execute Agent")
print("=" * 70)
pe_result = plan_execute_agent.invoke({"task": "透肌精华"})

print(f"\n任务: {MARKETING_TASK}")
print(f"\n计划 (Plan阶段一次性生成):")
for i, step in enumerate(pe_result["plan"]):
    print(f"  {i+1}. {step}")
print(f"\n执行 (Execute阶段顺序执行):")
for i, (step, result) in enumerate(zip(pe_result["plan"], pe_result["results"])):
    print(f"  Step {i+1}: {step}")
    print(f"          -> {result}")

# === 对比表 ===
print("\n" + "=" * 70)
print("ReAct vs Plan-Execute 对比表（同一个营销任务）")
print("=" * 70)
print(f"{'维度':<20} {'ReAct':<25} {'Plan-Execute':<25}")
print("-" * 70)
print(f"{'框架':<20} {'LangGraph':<25} {'LangGraph':<25}")
print(f"{'核心API':<20} {'create_react_agent':<25} {'StateGraph':<25}")
print(f"{'模型调用次数':<20} {react_model.call_index:<25} {'N/A (无LLM)':<25}")
print(f"{'工具调用次数':<20} {react_tool_calls:<25} {len(pe_result['results']):<25}")
print(f"{'Thought数':<20} {react_thoughts:<25} {'N/A (无推理)':<25}")
print(f"{'计划步数':<20} {'动态涌现':<25} {len(pe_result['plan']):<25}")
print(f"{'执行模式':<20} {'边推理边执行':<25} {'先规划后执行':<25}")
print(f"{'灵活性':<20} {'高（每步可调整）':<25} {'低（计划锁定）':<25}")
print(f"{'成本可预测性':<20} {'低':<25} {'高':<25}")
print(f"{'适用营销场景':<20} {'信息不足需探索':<25} {'信息充分结构化':<25}")


---
## TODO5：用CrewAI API结构编写等价实现（静态对比）

CrewAI采用"Agent即角色"设计哲学：
- 开发者定义Agent（role/goal/backstory）和Task（description/expected_output/agent）
- CrewAI根据Task的context依赖自动编排执行顺序
- 角色化协作，代码简洁

> ⚠️ 本环境未安装crewai。按v5.0规则不pip install，采用**静态API结构对比**：
> 编写真实CrewAI API代码（可读性等同实跑），用try/except ImportError处理，
> 代码结构真实反映CrewAI设计哲学。

> 参考教材 § Day 2 一、CrewAI设计哲学 + 二、双框架实现


In [ ]:
# TODO5: 用CrewAI API结构编写等价实现（静态对比）

print("=" * 70)
print("【静态对比】CrewAI 等价实现（Agent即角色）")
print("=" * 70)

try:
    from crewai import Agent, Task, Crew, Process
    crewai_available = True
    print("crewai已安装，实跑模式")
except ImportError:
    crewai_available = False
    print("crewai未安装 -> 静态API结构对比（不阻塞，按v5.0规则）")
    print("  代码结构真实反映CrewAI设计哲学，可读性等同实跑")

# === CrewAI API 结构（真实代码，未安装时仅展示结构）===
crewai_code = '''
from crewai import Agent, Task, Crew, Process

# 1. 定义Agent（角色化：role + goal + backstory）
product_researcher = Agent(
    role='产品调研专家',
    goal='深入分析透肌精华的产品功能和特点',
    backstory='你是一位有10年经验的产品分析师，擅长拆解护肤品的产品策略。',
    tools=[search_product_info],
    llm=llm
)

competitor_analyst = Agent(
    role='竞品分析专家',
    goal='分析雅诗兰黛的竞品策略',
    backstory='你是一位竞品分析专家，擅长拆解竞品定价和营销策略。',
    tools=[analyze_competitor],
    llm=llm
)

report_writer = Agent(
    role='竞品分析报告撰写人',
    goal='将调研结果整合成营销策略报告',
    backstory='你是一位商业分析师，擅长将零散信息整合为结构化报告。',
    llm=llm
)

# 2. 定义Task（context依赖隐式编排执行顺序）
product_task = Task(
    description='分析透肌精华的产品功能、核心卖点和差异化特性',
    expected_output='产品功能分析文档',
    agent=product_researcher
)

competitor_task = Task(
    description='分析雅诗兰黛的定价方案和营销策略',
    expected_output='竞品策略分析文档',
    agent=competitor_analyst
)

report_task = Task(
    description='将前两个调研结果整合为完整的营销策略报告',
    expected_output='结构化营销策略报告',
    agent=report_writer,
    context=[product_task, competitor_task]  # 依赖前两个任务
)

# 3. 组建Crew并执行
crew = Crew(
    agents=[product_researcher, competitor_analyst, report_writer],
    tasks=[product_task, competitor_task, report_task],
    process=Process.sequential,
    verbose=True
)

result = crew.kickoff(inputs={'competitor': '雅诗兰黛'})
'''

print("\nCrewAI 等价代码结构:")
print(crewai_code)

# === CrewAI vs LangGraph 设计哲学对比 ===
print("=" * 70)
print("CrewAI vs LangGraph 设计哲学对比")
print("=" * 70)
print(f"{'维度':<20} {'LangGraph':<25} {'CrewAI':<25}")
print("-" * 70)
print(f"{'设计哲学':<20} {'Agent即图':<25} {'Agent即角色':<25}")
print(f"{'核心抽象':<20} {'StateGraph':<25} {'Crew+Agent+Task':<25}")
print(f"{'控制流':<20} {'显式定义图结构':<25} {'Task context隐式编排':<25}")
print(f"{'角色定义':<20} {'无（节点是函数）':<25} {'role+goal+backstory':<25}")
print(f"{'任务分配':<20} {'节点边定义':<25} {'Task.agent显式分配':<25}")
print(f"{'依赖管理':<20} {'add_edge显式':<25} {'Task.context隐式':<25}")
print(f"{'并行能力':<20} {'原生fan-out/fan-in':<25} {'需Process.hierarchical':<25}")
print(f"{'代码量':<20} {'约50行':<25} {'约45行':<25}")
print(f"{'学习曲线':<20} {'陡峭':<25} {'平缓':<25}")
print(f"{'适用营销场景':<20} {'复杂工作流精确控制':<25} {'角色明确团队协作':<25}")
print(f"{'本Day状态':<20} {'真实运行':<25} {'静态API对比':<25}")

print(f"\n CrewAI选型建议: 营销任务可清晰分解为角色职责（调研员/分析师/撰写人）时选CrewAI")
print(f" LangGraph选型建议: 需要精确控制流程、条件分支、HITL时选LangGraph")


---
## TODO6：用AutoGen API结构编写等价实现 + 四框架对比表

AutoGen采用"Agent即对话者"设计哲学：
- 每个Agent是ConversableAgent，可发送和接收消息
- 通过GroupChat机制，多个Agent在同一个对话中交互
- 适合需要Agent间讨论和协商的场景

> ⚠️ 本环境未安装autogen。采用**静态API结构对比**。

最后生成**四框架对比表**（LangGraph/CrewAI/AutoGen/MetaGPT），
作为本Day的核心交付物。

> 参考教材 § Day 2 三、AutoGen和MetaGPT的适用场景


In [ ]:
# TODO6: 用AutoGen API结构编写等价实现 + 四框架对比表

print("=" * 70)
print("【静态对比】AutoGen 等价实现（Agent即对话者）")
print("=" * 70)

try:
    from autogen import ConversableAgent, GroupChat, GroupChatManager
    autogen_available = True
    print("autogen已安装，实跑模式")
except ImportError:
    autogen_available = False
    print("autogen未安装 -> 静态API结构对比（不阻塞，按v5.0规则）")
    print("  代码结构真实反映AutoGen设计哲学，可读性等同实跑")

# === AutoGen API 结构（真实代码，未安装时仅展示结构）===
autogen_code = '''
from autogen import ConversableAgent, GroupChat, GroupChatManager

# 1. 定义ConversableAgent（对话驱动）
product_researcher = ConversableAgent(
    name="product_researcher",
    system_message="你是产品调研专家，负责分析透肌精华的产品功能。只讨论产品维度。"
)

competitor_analyst = ConversableAgent(
    name="competitor_analyst",
    system_message="你是竞品分析专家，负责分析雅诗兰黛的策略。只讨论竞品维度。"
)

decision_maker = ConversableAgent(
    name="decision_maker",
    system_message="你是营销决策者，综合产品调研员和竞品分析师的意见，制定最终策略。"
)

# 2. 组建GroupChat（多Agent对话）
group_chat = GroupChat(
    agents=[product_researcher, competitor_analyst, decision_maker],
    messages=[],
    max_round=10  # 最多讨论10轮，防止无限对话
)

manager = GroupChatManager(group_chat)

# 3. 发起对话（对话驱动执行）
decision_maker.initiate_chat(
    manager,
    message="我们需要为透肌精华制定营销策略，竞品是雅诗兰黛。请各位发表意见。"
)
'''

print("\nAutoGen 等价代码结构:")
print(autogen_code)

# === 四框架对比表（本Day核心交付物）===
print("\n" + "=" * 70)
print("四框架对比表（同一个营销任务，Agent框架选型核心参考）")
print("=" * 70)
print(f"{'维度':<16} {'LangGraph':<14} {'CrewAI':<14} {'AutoGen':<14} {'MetaGPT':<14}")
print("-" * 70)
print(f"{'设计哲学':<16} {'Agent即图':<14} {'Agent即角色':<14} {'Agent即对话者':<14} {'Agent即流程':<14}")
print(f"{'核心抽象':<16} {'StateGraph':<14} {'Crew+Task':<14} {'GroupChat':<14} {'SOP':<14}")
print(f"{'控制流':<16} {'显式图':<14} {'Task编排':<14} {'对话驱动':<14} {'预定义SOP':<14}")
print(f"{'灵活性':<16} {'极高':<14} {'中高':<14} {'中':<14} {'中低':<14}")
print(f"{'学习曲线':<16} {'陡峭':<14} {'平缓':<14} {'中等':<14} {'中等':<14}")
print(f"{'状态管理':<16} {'State类型安全':<14} {'Task context':<14} {'对话历史':<14} {'SOP协议':<14}")
print(f"{'HITL':<16} {'原生支持':<14} {'需自定义':<14} {'需自定义':<14} {'需自定义':<14}")
print(f"{'并行':<16} {'原生fan-out':<14} {'hierarchical':<14} {'对话并发':<14} {'SOP阶段':<14}")
print(f"{'维护方':<16} {'LangChain':<14} {'CrewAI':<14} {'微软':<14} {'DeepWisdom':<14}")
print(f"{'适用场景':<16} {'复杂工作流':<14} {'角色协作':<14} {'讨论协商':<14} {'标准化流程':<14}")
print(f"{'营销适用':<16} {'精确控制':<14} {'团队分工':<14} {'多视角辩论':<14} {'稳定输出':<14}")
print(f"{'本Day状态':<16} {'真实运行':<14} {'静态对比':<14} {'静态对比':<14} {'文档对比':<14}")

# === 框架选择决策树 ===
print("\n" + "=" * 70)
print("框架选择决策树（教材 § Day 2 三节）")
print("=" * 70)
print("1. 需要精确控制执行流程？        -> LangGraph")
print("2. 任务可以按角色分工？          -> CrewAI")
print("3. 需要Agent间讨论和辩论？       -> AutoGen")
print("4. 有标准化的SOP需要遵循？       -> MetaGPT")
print("5. 以上都不满足，需要混合方案？  -> LangGraph骨架 + 关键节点嵌入CrewAI/AutoGen")

# === 本Day总结 ===
print("\n" + "=" * 70)
print("本Day总结：同一个营销任务，不同框架的因果链差异")
print("=" * 70)
print(" LangGraph ReAct:       因果链在每步涌现（Thought->Action->Obs循环）")
print(" LangGraph Plan-Execute: 因果链在Plan阶段确定（Execute不可调整）")
print(" CrewAI:                因果链由Task context依赖隐式编排（角色分工）")
print(" AutoGen:               因果链在对话中涌现（多轮讨论->决策）")
print(" MetaGPT:               因果链由SOP预定义（流程固定）")
print("\n 天道推演视角: 框架选型 = 选择不同的因果网络结构")
print(" 营销任务（透肌精华竞品分析）选型建议: 信息充分->Plan-Execute，需多视角->AutoGen")
